In [88]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [
        18, 19, 20, 21, 22, 23, 24, 25, 20, 21,
        22, 23, 24, 26, 19, 20, 21, 22, 25, 27,
        18, 23, 24, 21, 26, 20, 22, 25, 19, 24
    ],

    "hours": [
        1, 2, 3, 4, 5, 6, 7, 8, 3, 4,
        5, 6, 7, 9, 2, np.nan, 4, 5, 8, 9,
        1, 6, 7, 4, 8, 3, np.nan, 8, 2, 7
    ],

    "sleep": [
        8, 8, 7, 7, 7, 6, 6, 5, 8, 7,
        7, 6, 6, 5, 8, 7, 7, 6, 6, 5,
        9, 6, 5, 7, 5, 8, 6, 5, 8, 6
    ],

    "city": [
        "Moscow", "Berlin", "Moscow", "Paris", "Berlin",
        "Moscow", "Paris", "Berlin", "Moscow", "Paris",
        "Berlin", "Moscow", "Paris", "Berlin", "Moscow",
        "Paris", "Berlin", "Moscow", "Paris", "Berlin",
        "Moscow", "Paris", "Berlin", "Moscow", "Paris",
        "Berlin", "Moscow", "Paris", "Berlin", "Moscow"
    ],

    "passed": [
        0, 0, 0, 0, 1, 1, 1, 1, 0, 0,
        1, 1, 1, 1, 0, 0, 0, 1, 1, 1,
        0, 1, 1, 0, 1, 0, 1, 1, 0, 1
    ]
})

***Тип задачи: классификация***
***Таргет:passed***
***Числовые признаки: age hours sleep***
***Категориальный признаки: city***

In [89]:
print(df.info())
print("\nDuplicase:", df.duplicated().sum())
print("\nTarget balance:", df["passed"].value_counts(normalize=True))

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     30 non-null     int64  
 1   hours   28 non-null     float64
 2   sleep   30 non-null     int64  
 3   city    30 non-null     str    
 4   passed  30 non-null     int64  
dtypes: float64(1), int64(3), str(1)
memory usage: 1.3 KB
None

Duplicase: 5

Target balance: passed
1    0.566667
0    0.433333
Name: proportion, dtype: float64


In [90]:
df.isna().sum()

age       0
hours     2
sleep     0
city      0
passed    0
dtype: int64

Получили, что у датасета 2 пропуска у hours, есть только 5 дубликатов, таргет распределен практически идеально по классам


In [91]:
X = df.drop(columns="passed")
Y = df["passed"]
print(X,Y)
print(X.shape,Y.shape)


    age  hours  sleep    city
0    18    1.0      8  Moscow
1    19    2.0      8  Berlin
2    20    3.0      7  Moscow
3    21    4.0      7   Paris
4    22    5.0      7  Berlin
5    23    6.0      6  Moscow
6    24    7.0      6   Paris
7    25    8.0      5  Berlin
8    20    3.0      8  Moscow
9    21    4.0      7   Paris
10   22    5.0      7  Berlin
11   23    6.0      6  Moscow
12   24    7.0      6   Paris
13   26    9.0      5  Berlin
14   19    2.0      8  Moscow
15   20    NaN      7   Paris
16   21    4.0      7  Berlin
17   22    5.0      6  Moscow
18   25    8.0      6   Paris
19   27    9.0      5  Berlin
20   18    1.0      9  Moscow
21   23    6.0      6   Paris
22   24    7.0      5  Berlin
23   21    4.0      7  Moscow
24   26    8.0      5   Paris
25   20    3.0      8  Berlin
26   22    NaN      6  Moscow
27   25    8.0      5   Paris
28   19    2.0      8  Berlin
29   24    7.0      6  Moscow 0     0
1     0
2     0
3     0
4     1
5     1
6     1
7     1
8     

Просмотрел X и Y, все распределилось првильно. Размеры корректны и такие размеры у таргета, потому что это просто вектор в D-1 пространстве

In [92]:
from sklearn.model_selection import train_test_split

X_train_city, X_test_city, Y_train_city, Y_test_city = train_test_split(X, Y, test_size= 0.2, random_state=42, stratify=Y)

print(X_train_city.shape, X_test_city.shape, Y_train_city.shape, Y_test_city.shape)

(24, 4) (6, 4) (24,) (6,)


In [93]:
print(Y.value_counts(normalize=True))
print(Y_train_city.value_counts(normalize=True))
print(Y_test_city.value_counts(normalize=True))

passed
1    0.566667
0    0.433333
Name: proportion, dtype: float64
passed
1    0.583333
0    0.416667
Name: proportion, dtype: float64
passed
0    0.5
1    0.5
Name: proportion, dtype: float64


Нужно использовать stratify потому что класс может не равномерно быть распределен между классамо


In [94]:
from sklearn.impute import SimpleImputer

X_num = df[["age", "hours", "sleep"]]
Y = df["passed"]

X_train, X_test, Y_train, Y_test = train_test_split(X_num, Y, test_size= 0.2, random_state= 42, stratify=Y)

Imputer = SimpleImputer(strategy="median")
Imputer.fit(X_train)

X_train_imputed = Imputer.transform(X_train)
X_test_imputed = Imputer.transform(X_test)



In [95]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train_imputed)

X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print(X_train_scaled.mean(axis=0))
print(X_train_scaled.std(axis=0))

[-4.71844785e-16 -1.29526020e-16  2.22044605e-16]
[1. 1. 1.]


In [96]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, Y_train)

Y_pred = model.predict(X_test)

/opt/homebrew/Caskroom/miniconda/base/envs/ml_learn/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [97]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(Y_test, Y_pred)
print(accuracy)

0.5


In [98]:
Y_proba = model.predict_proba(X_test_scaled)

print(Y_proba)
print(Y_proba.shape)

[[0.62210156 0.37789844]
 [0.00267857 0.99732143]
 [0.96219924 0.03780076]
 [0.03987263 0.96012737]
 [0.62210156 0.37789844]
 [0.03987263 0.96012737]]
(6, 2)


In [99]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipeline.fit(X_train, Y_train)

Y_pred_pipeline = pipeline.predict(X_test)

accuracy_pipeline = accuracy_score(Y_test, Y_pred_pipeline)

print(accuracy_pipeline)

1.0


In [103]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_features = ["age", "hours", "sleep"]
categorical_features = ["city"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

final_model.fit(X_train_city, Y_train_city)

Y_pred_final = final_model.predict(X_test_city)

print(accuracy_score(Y_test_city, Y_pred_final))


1.0
